# Context Engineering: Compaction, Tool Clearing, and Memory

The full code of [Context Engineering: Compaction, Tool Clearing, and Memory](https://residual-stream.lvtan71.workers.dev/posts/context-engineering-compaction-clearing-memory/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/context-engineering-compaction-clearing-memory) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/context-engineering-compaction-clearing-memory.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 20 min end to end on the Apple M4 that saved the outputs below.
Six of the eight scripts call the Anthropic API and need ANTHROPIC_API_KEY. The post ran claude-sonnet-5-5 (about $11 at list prices); the saved outputs here ran claude-haiku-5-5 (set MODEL=claude-haiku-5-5), for about $1.20, so their turns, summaries and context sizes differ from the post's. The model is not deterministic either way.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on an Apple M4 on 2026-10-18 with `MODEL=claude-haiku-5-5`** (anthropic 1.13, 20 min, about $1.20 of API use; every cell ran). The post's numbers come from claude-sonnet-5-5, so these turns, summaries and context sizes differ. Run the notebook to replace them with your own.

In [1]:
%pip install -q "anthropic==1.13.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "context-engineering-compaction-clearing-memory"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/context-engineering-compaction-clearing-memory


## `scratch.py`

Clearing as a pass over a message list, and the memory backend's path guard, without the API.

In [2]:
!{sys.executable} scratch.py

clearing, keep=1: 193,182 -> 48,501 characters
tool_use records kept: ['/research/celegans_review.md', '/research/drosophila_review.md', '/research/yeast_review.md', '/research/killifish_review.md']
Created /memories/notes.md


Error: path escapes the memory directory: /memories/../../escape.txt


The post printed:

```text
clearing, keep=1: 193,182 -> 48,501 characters
tool_use records kept: ['/research/celegans_review.md', '/research/drosophila_review.md', '/research/yeast_review.md', '/research/killifish_review.md']
Created /memories/notes.md
Error: path escapes the memory directory: /memories/../../escape.txt
```

## `kvcost.py`

What the measured context sizes would hold in a KV cache for three open models.

In [3]:
!{sys.executable} kvcost.py

Llama 2 7B (32 layers, 32 KV heads of 128): 512 KiB per token
   this post's baseline peak, 90,421 tokens:   44.2 GiB
   the cookbook's baseline peak, 335,279 tokens:  163.7 GiB
Llama 2 70B (80 layers, 8 KV heads of 128): 320 KiB per token
   this post's baseline peak, 90,421 tokens:   27.6 GiB
   the cookbook's baseline peak, 335,279 tokens:  102.3 GiB
Qwen2.5-1.5B (28 layers, 2 KV heads of 128): 28 KiB per token
   this post's baseline peak, 90,421 tokens:    2.4 GiB
   the cookbook's baseline peak, 335,279 tokens:    9.0 GiB


The post printed:

```text
Llama 2 7B (32 layers, 32 KV heads of 128): 512 KiB per token
   this post's baseline peak, 90,421 tokens:   44.2 GiB
   the cookbook's baseline peak, 335,279 tokens:  163.7 GiB
Llama 2 70B (80 layers, 8 KV heads of 128): 320 KiB per token
   this post's baseline peak, 90,421 tokens:   27.6 GiB
   the cookbook's baseline peak, 335,279 tokens:  102.3 GiB
Qwen2.5-1.5B (28 layers, 2 KV heads of 128): 28 KiB per token
   this post's baseline peak, 90,421 tokens:    2.4 GiB
   the cookbook's baseline peak, 335,279 tokens:    9.0 GiB
```

## `baseline.py`

The agent with no context management, and what fills its window.

In [4]:
!{sys.executable} baseline.py

[no management]


  turn  1  context     652  search_files, search_files


  turn  2  context   1,001  read_file, read_file, read_file, read_file, read_file, read_file


  turn  3  context  89,768  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding


  turn  4  context  90,917  final answer


corpus: 6 documents, 88,317 tokens
peak context 90,917 tokens after 4 turns and 6 file reads
  file-read results   88,317   97.1%
  everything else      2,600    2.9%
API usage: 182,338 uncached input, 0 cache-write, 0 cache-read, 2,096 output tokens; $0.02


The post printed:

```text
[no management]
  turn  1  context     652  search_files
  turn  2  context     780  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context  89,529  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding
  turn  4  context  90,421  final answer
corpus: 6 documents, 88,317 tokens
peak context 90,421 tokens after 4 turns and 6 file reads
  file-read results   88,317   97.7%
  everything else      2,104    2.3%
API usage: 181,382 uncached input, 0 cache-write, 0 cache-read, 1,800 output tokens; $0.38
```

## `clearing.py`

Tool-result clearing with keep = 2, 4 and 6, three runs of each setting that clears.

In [5]:
!{sys.executable} clearing.py

[clearing, keep 2, run 1]


  turn  1  context     652  search_files, search_files


  turn  2  context   1,008  read_file, read_file, read_file, read_file, read_file, read_file


  turn  3  context  30,821  read_file, read_file, read_file, read_file  clearing


  turn  4  context  31,345  read_file, read_file, read_file, read_file  clearing


...

[clearing, keep 6, run 1]


  turn  1  context     652  search_files, search_files


  turn  2  context   1,008  read_file, read_file, read_file, read_file, read_file, read_file


  turn  3  context  89,773  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding


  turn  4  context  90,979  final answer
keep 6: finished after 4 turns; peak context 90,979; 6 file reads, 0 documents read more than once
API usage: 2,615,474 uncached input, 0 cache-write, 0 cache-read, 33,076 output tokens; $0.28


The post printed:

```text
[clearing, keep 2, run 1]
  turn  1  context     652  search_files
  turn  2  context     780  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context  30,658  read_file, read_file, read_file, read_file, record_finding, record_finding  clearing
  turn  4  context  31,542  record_finding, record_finding, read_file, read_file  clearing
  turn  5  context  32,201  record_finding, record_finding  clearing
  turn  6  context  32,516  final answer  clearing
keep 2: finished after 6 turns; peak context 32,516; 12 file reads, 4 documents read more than once
  turn  3: cleared_tool_uses = 5, cleared_input_tokens = 59,014
  turn  4: cleared_tool_uses = 9, cleared_input_tokens = 117,990
  turn  5: cleared_tool_uses = 11, cleared_input_tokens = 147,508
  turn  6: cleared_tool_uses = 11, cleared_input_tokens = 147,569
[clearing, keep 2, run 2]
  turn  1  context     652  search_files
  turn  2  context     780  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context  30,658  read_file, read_file, read_file, read_file  clearing
  turn  4  context  31,171  record_finding, record_finding, read_file, read_file, read_file, read_file  clearing
  turn  5  context  32,100  record_finding, record_finding, read_file, read_file  clearing
  turn  6  context  32,778  record_finding, record_finding  clearing
  turn  7  context  33,117  final answer  clearing
keep 2: finished after 7 turns; peak context 33,117; 16 file reads, 6 documents read more than once
  turn  3: cleared_tool_uses = 5, cleared_input_tokens = 59,014
  turn  4: cleared_tool_uses = 9, cleared_input_tokens = 117,931
  turn  5: cleared_tool_uses = 13, cleared_input_tokens = 176,902
  turn  6: cleared_tool_uses = 15, cleared_input_tokens = 206,422
  turn  7: cleared_tool_uses = 15, cleared_input_tokens = 206,478
[clearing, keep 2, run 3]
  turn  1  context     652  search_files
  turn  2  context     780  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context  30,658  read_file, read_file, read_file, read_file  clearing
  turn  4  context  31,173  record_finding, record_finding, read_file, read_file, read_file, read_file  clearing
  turn  5  context  31,927  record_finding, record_finding, read_file  clearing
  turn  6  context  32,466  record_finding, read_file  clearing
  turn  7  context  32,720  record_finding  clearing
  turn  8  context  32,851  final answer  clearing
keep 2: finished after 8 turns; peak context 32,851; 16 file reads, 6 documents read more than once
  turn  3: cleared_tool_uses = 5, cleared_input_tokens = 59,014
  turn  4: cleared_tool_uses = 9, cleared_input_tokens = 117,931
  turn  5: cleared_tool_uses = 13, cleared_input_tokens = 176,884
  turn  6: cleared_tool_uses = 14, cleared_input_tokens = 191,666
  turn  7: cleared_tool_uses = 15, cleared_input_tokens = 206,410
  turn  8: cleared_tool_uses = 15, cleared_input_tokens = 206,436
[clearing, keep 4, run 1]
  turn  1  context     652  search_files
  turn  2  context     780  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context  60,067  read_file, read_file  clearing
  turn  4  context  60,368  record_finding, record_finding, record_finding, record_finding, read_file, read_file  clearing
  turn  5  context  61,327  record_finding, record_finding  clearing
  turn  6  context  61,640  final answer  clearing
keep 4: finished after 6 turns; peak context 61,640; 10 file reads, 4 documents read more than once
  turn  3: cleared_tool_uses = 3, cleared_input_tokens = 29,528
  turn  4: cleared_tool_uses = 5, cleared_input_tokens = 59,002
  turn  5: cleared_tool_uses = 7, cleared_input_tokens = 88,534
  turn  6: cleared_tool_uses = 7, cleared_input_tokens = 88,573
[clearing, keep 4, run 2]
  turn  1  context     652  search_files
  turn  2  context     780  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context  60,067  read_file, read_file  clearing
  turn  4  context  60,367  read_file, read_file  clearing
  turn  5  context  60,713  read_file, read_file  clearing
  turn  6  context  60,995  record_finding, record_finding, record_finding, record_finding, read_file, read_file  clearing
  turn  7  context  61,858  record_finding, record_finding  clearing
  turn  8  context  62,112  final answer  clearing
keep 4: finished after 8 turns; peak context 62,112; 14 file reads, 6 documents read more than once
  turn  3: cleared_tool_uses = 3, cleared_input_tokens = 29,528
  turn  4: cleared_tool_uses = 5, cleared_input_tokens = 59,002
  turn  5: cleared_tool_uses = 7, cleared_input_tokens = 88,471
  turn  6: cleared_tool_uses = 9, cleared_input_tokens = 117,940
  turn  7: cleared_tool_uses = 11, cleared_input_tokens = 147,488
  turn  8: cleared_tool_uses = 11, cleared_input_tokens = 147,534
[clearing, keep 4, run 3]
  turn  1  context     652  search_files
  turn  2  context     780  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context  60,067  read_file, read_file  clearing
  turn  4  context  60,374  read_file, read_file  clearing
  turn  5  context  60,747  record_finding, record_finding, record_finding, record_finding, read_file, read_file  clearing
  turn  6  context  61,675  record_finding, record_finding  clearing
  turn  7  context  61,960  final answer  clearing
keep 4: finished after 7 turns; peak context 61,960; 12 file reads, 6 documents read more than once
  turn  3: cleared_tool_uses = 3, cleared_input_tokens = 29,528
  turn  4: cleared_tool_uses = 5, cleared_input_tokens = 59,002
  turn  5: cleared_tool_uses = 7, cleared_input_tokens = 88,489
  turn  6: cleared_tool_uses = 9, cleared_input_tokens = 118,038
  turn  7: cleared_tool_uses = 9, cleared_input_tokens = 118,082
[clearing, keep 6, run 1]
  turn  1  context     652  search_files
  turn  2  context     780  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context  89,529  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding
  turn  4  context  90,591  final answer
keep 6: finished after 4 turns; peak context 90,591; 6 file reads, 0 documents read more than once
API usage: 1,583,017 uncached input, 0 cache-write, 0 cache-read, 17,094 output tokens; $3.34
```

## `compaction.py`

Compaction with the default prompt and with instructions, on the plain and the dense corpus, three runs each, each summary probed. Requests allow 16,000 output tokens, since the summary is written within them.

In [6]:
!{sys.executable} compaction.py

plain corpus: 88,317 tokens
[compaction, plain corpus, default prompt, run 1]


  turn  1  context     694  search_files, read_file


  turn  2  context   1,002  read_file, read_file, read_file, read_file, read_file, read_file


  turn  3  context   1,766  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction


  turn  4  context   3,016  final answer


...

  turn  6  context   5,918  read_file


  turn  7  context  27,918  search_files


  turn  8  context  31,846  record_finding, record_finding


  turn  9  context  32,372  final answer


  130,669 tokens -> 2,265-token summary (58x, 5,202 output tokens billed) at turn 3, final context 32,372; from the summary alone: lifespans 6/6, source identifiers 6/6
API usage: 2,172,446 uncached input, 0 cache-write, 0 cache-read, 144,305 output tokens; $0.69


The post printed:

```text
plain corpus: 88,317 tokens
[compaction, plain corpus, default prompt, run 1]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,548  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,726  final answer
  89,615 tokens -> 885-token summary (101x, 889 output tokens billed) at turn 3, final context 2,726; from the summary alone: lifespans 6/6, source identifiers 6/6
[compaction, plain corpus, default prompt, run 2]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,671  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,968  final answer
  89,615 tokens -> 1,008-token summary (89x, 1,012 output tokens billed) at turn 3, final context 2,968; from the summary alone: lifespans 6/6, source identifiers 6/6
[compaction, plain corpus, default prompt, run 3]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,484  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,696  final answer
  89,615 tokens -> 821-token summary (109x, 825 output tokens billed) at turn 3, final context 2,696; from the summary alone: lifespans 6/6, source identifiers 6/6
[compaction, plain corpus, guided prompt, run 1]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,564  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,566  final answer
  89,664 tokens -> 901-token summary (100x, 1,270 output tokens billed) at turn 3, final context 2,566; from the summary alone: lifespans 6/6, source identifiers 6/6
[compaction, plain corpus, guided prompt, run 2]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,556  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,731  final answer
  89,664 tokens -> 893-token summary (100x, 1,202 output tokens billed) at turn 3, final context 2,731; from the summary alone: lifespans 6/6, source identifiers 6/6
[compaction, plain corpus, guided prompt, run 3]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,779  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,977  final answer
  89,664 tokens -> 1,116-token summary (80x, 1,458 output tokens billed) at turn 3, final context 2,977; from the summary alone: lifespans 6/6, source identifiers 6/6
dense corpus: 129,152 tokens
[compaction, dense corpus, default prompt, run 1]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,840  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   3,239  final answer
  130,450 tokens -> 1,177-token summary (111x, 1,181 output tokens billed) at turn 3, final context 3,239; from the summary alone: lifespans 6/6, source identifiers 0/6
[compaction, dense corpus, default prompt, run 2]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,879  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   3,450  final answer
  130,450 tokens -> 1,216-token summary (107x, 1,220 output tokens billed) at turn 3, final context 3,450; from the summary alone: lifespans 6/6, source identifiers 0/6
[compaction, dense corpus, default prompt, run 3]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,922  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   3,539  final answer
  130,450 tokens -> 1,259-token summary (104x, 1,263 output tokens billed) at turn 3, final context 3,539; from the summary alone: lifespans 6/6, source identifiers 0/6
[compaction, dense corpus, guided prompt, run 1]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   3,348  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   4,894  final answer
  130,499 tokens -> 2,685-token summary (49x, 4,693 output tokens billed) at turn 3, final context 4,894; from the summary alone: lifespans 6/6, source identifiers 6/6
[compaction, dense corpus, guided prompt, run 2]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   3,516  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   5,094  final answer
  130,499 tokens -> 2,853-token summary (46x, 4,943 output tokens billed) at turn 3, final context 5,094; from the summary alone: lifespans 6/6, source identifiers 6/6
[compaction, dense corpus, guided prompt, run 3]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   3,191  search_files, search_files  compaction
  turn  4  context   3,738  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding
  turn  5  context   5,253  final answer
  130,499 tokens -> 2,528-token summary (52x, 4,167 output tokens billed) at turn 3, final context 5,253; from the summary alone: lifespans 6/6, source identifiers 6/6
API usage: 1,431,371 uncached input, 0 cache-write, 0 cache-read, 54,174 output tokens; $3.40
```

## `repeat.py`

Ten more runs of compaction on the plain corpus with the default prompt.

In [7]:
!{sys.executable} repeat.py

[plain corpus, default prompt, run 1]


  turn  1  context     694  search_files, search_files


  turn  2  context   1,044  read_file, read_file, read_file, read_file, read_file, read_file


  turn  3  context   2,012  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction


  turn  4  context   3,179  record_finding


...

  turn  1  context     694  search_files, search_files


  turn  2  context   1,041  read_file, read_file, read_file, read_file, read_file, read_file


  turn  3  context   1,825  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction


  turn  4  context   3,334  final answer


  lifespans 6/6, source identifiers 6/6
summaries that kept every identifier: 10 of 10
API usage: 983,265 uncached input, 0 cache-write, 0 cache-read, 41,221 output tokens; $0.12


The post printed:

```text
[plain corpus, default prompt, run 1]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,556  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,453  final answer
  lifespans 6/6, source identifiers 6/6
[plain corpus, default prompt, run 2]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,622  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,750  final answer
  lifespans 6/6, source identifiers 6/6
[plain corpus, default prompt, run 3]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,532  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,518  final answer
  lifespans 6/6, source identifiers 6/6
[plain corpus, default prompt, run 4]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,727  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,618  final answer
  lifespans 6/6, source identifiers 6/6
[plain corpus, default prompt, run 5]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,471  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,367  final answer
  lifespans 6/6, source identifiers 6/6
[plain corpus, default prompt, run 6]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,541  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,612  final answer
  lifespans 6/6, source identifiers 0/6
[plain corpus, default prompt, run 7]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,700  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   3,029  final answer
  lifespans 6/6, source identifiers 6/6
[plain corpus, default prompt, run 8]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,544  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,641  final answer
  lifespans 6/6, source identifiers 6/6
[plain corpus, default prompt, run 9]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,586  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,802  final answer
  lifespans 6/6, source identifiers 6/6
[plain corpus, default prompt, run 10]
  turn  1  context     694  search_files
  turn  2  context     822  read_file, read_file, read_file, read_file, read_file, read_file
  turn  3  context   1,451  record_finding, record_finding, record_finding, record_finding, record_finding, record_finding  compaction
  turn  4  context   2,692  final answer
  lifespans 6/6, source identifiers 6/6
summaries that kept every identifier: 9 of 10
API usage: 965,942 uncached input, 0 cache-write, 0 cache-read, 30,661 output tokens; $2.24
```

## `memory.py`

Two sessions with the memory tool; the second run with and without the first one's notes.

In [8]:
!{sys.executable} memory.py

[session 1]


  turn  1  context   2,113  memory, read_file, read_file, read_file


  turn  2  context  46,593  memory


  turn  3  context  47,732  record_finding


  turn  4  context  47,927  final answer


...

  turn  2  context   2,333  memory


  turn  3  context   3,257  read_file, read_file, read_file


  turn  4  context  47,705  memory


  turn  5  context  48,402  memory


  turn  6  context  48,854  final answer
session 2 with memory: 3 file reads, 6 turns, peak 48,854
API usage: 577,707 uncached input, 0 cache-write, 0 cache-read, 7,277 output tokens; $0.06


The post printed:

```text
[session 1]
  turn  1  context   2,113  memory
  turn  2  context   2,191  read_file, read_file, read_file
  turn  3  context  46,577  memory, record_finding
  turn  4  context  47,431  final answer
session 1: 3 file reads, peak 47,431; saved 1 file(s), 525 tokens
[session 2, without memory]
  turn  1  context   2,081  memory
  turn  2  context   2,159  search_files, search_files
  turn  3  context   2,471  read_file, read_file, read_file, read_file, read_file, read_file
  turn  4  context  91,220  memory
  turn  5  context  91,650  final answer
session 2 without memory: 6 file reads, 5 turns, peak 91,650
[session 2, with memory]
  turn  1  context   2,081  memory
  turn  2  context   2,169  memory, search_files
  turn  3  context   3,006  read_file, read_file, read_file
  turn  4  context  47,386  memory
  turn  5  context  48,298  final answer
session 2 with memory: 3 file reads, 5 turns, peak 48,298
API usage: 390,833 uncached input, 0 cache-write, 0 cache-read, 5,354 output tokens; $0.84
```

## `combined.py`

All three together.

In [9]:
!{sys.executable} combined.py

[all three]


  turn  1  context   2,145  memory, search_files


  turn  2  context   2,391  read_file, read_file, read_file, read_file, read_file, read_file


  turn  3  context  32,297  memory, read_file, read_file, read_file, read_file  clearing


  turn  4  context  33,293  memory, read_file, read_file  clearing


  turn  5  context  34,066  memory  clearing


  turn  6  context  34,657  final answer  clearing
peak context 34,657 after 6 turns and 12 file reads; 4 clearing pass(es), 0 compaction(s)
API usage: 138,849 uncached input, 0 cache-write, 0 cache-read, 3,701 output tokens; $0.02


The post printed:

```text
[all three]
  turn  1  context   2,145  memory
  turn  2  context   2,223  search_files
  turn  3  context   2,351  read_file, read_file, read_file, read_file, read_file, read_file
  turn  4  context  32,229  memory, record_finding, record_finding, read_file  clearing
  turn  5  context  33,102  memory, record_finding, read_file  clearing
  turn  6  context  33,553  memory, record_finding, read_file  clearing
  turn  7  context  33,986  memory, record_finding, read_file  clearing
  turn  8  context  34,441  record_finding, memory  clearing
  turn  9  context  34,831  final answer  clearing
peak context 34,831 after 9 turns and 10 file reads; 6 clearing pass(es), 0 compaction(s)
API usage: 208,861 uncached input, 0 cache-write, 0 cache-read, 3,146 output tokens; $0.45
```